In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

credentials_location = '/home/aryav/Projects/Airport-Dashboard/terraform/keys/airport-creds.json'
path_to_gcs_connector = "./lib/gcs-connector.jar"

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", path_to_gcs_connector) \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.recommender.gcs.GCSFileSystem")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)

spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/07 16:24:09 WARN Utils: Your hostname, aryav-Yoga-Pro-7-14IAH10, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlp1s0f0)
26/04/07 16:24:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/04/07 16:24:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
bucket_name="airport-dashboard-492216-bucket"

In [3]:
# Try reading the file you uploaded with Kestra
df = spark.read.csv(f"gs://{bucket_name}/raw/survey/2025/results.csv", header=True)

In [4]:
df.columns

['ResponseId',
 'MainBranch',
 'Age',
 'EdLevel',
 'Employment',
 'EmploymentAddl',
 'WorkExp',
 'LearnCodeChoose',
 'LearnCode',
 'LearnCodeAI',
 'AILearnHow',
 'YearsCode',
 'DevType',
 'OrgSize',
 'ICorPM',
 'RemoteWork',
 'PurchaseInfluence',
 'TechEndorseIntro',
 'TechEndorse_1',
 'TechEndorse_2',
 'TechEndorse_3',
 'TechEndorse_4',
 'TechEndorse_5',
 'TechEndorse_6',
 'TechEndorse_7',
 'TechEndorse_8',
 'TechEndorse_9',
 'TechEndorse_13',
 'TechEndorse_13_TEXT',
 'TechOppose_1',
 'TechOppose_2',
 'TechOppose_3',
 'TechOppose_5',
 'TechOppose_7',
 'TechOppose_9',
 'TechOppose_11',
 'TechOppose_13',
 'TechOppose_16',
 'TechOppose_15',
 'TechOppose_15_TEXT',
 'Industry',
 'JobSatPoints_1',
 'JobSatPoints_2',
 'JobSatPoints_3',
 'JobSatPoints_4',
 'JobSatPoints_5',
 'JobSatPoints_6',
 'JobSatPoints_7',
 'JobSatPoints_8',
 'JobSatPoints_9',
 'JobSatPoints_10',
 'JobSatPoints_11',
 'JobSatPoints_13',
 'JobSatPoints_14',
 'JobSatPoints_15',
 'JobSatPoints_16',
 'JobSatPoints_15_TEXT',
 

In [5]:
small_df = df.limit(1000)

In [6]:
small_df.select('Currency', 'ConvertedCompYearly', 'CompTotal').show(10)

[Stage 1:>                                                          (0 + 1) / 1]

+--------------------+-------------------+---------+
|            Currency|ConvertedCompYearly|CompTotal|
+--------------------+-------------------+---------+
|   EUR European Euro|              61256|    52800|
|   EUR European Euro|             104413|    90000|
|UAH Ukrainian hry...|              53061|  2214000|
|   EUR European Euro|              36197|    31200|
|USD United States...|              60000|    60000|
|USD United States...|             120000|   120000|
|USD United States...|               6240|     6240|
|USD United States...|              72000|    72000|
|USD United States...|              70000|    70000|
|USD United States...|               5400|     5400|
+--------------------+-------------------+---------+
only showing top 10 rows


In [7]:
from pyspark.sql import functions as F

In [8]:
small_df.select('YearsCode').show()

[Stage 2:>                                                          (0 + 1) / 1]

+---------+
|YearsCode|
+---------+
|       14|
|       10|
|       12|
|        5|
|       22|
|       20|
|       13|
|       30|
|       15|
|       10|
|        9|
|       NA|
|     NULL|
|     NULL|
|        5|
|       10|
|        6|
|       NA|
|        3|
|       40|
+---------+
only showing top 20 rows


In [36]:
numeric_regex = "^[0-9]+$"
float_regex = r"^[0-9]*\.?[0-9]+$"

df_tile1 = small_df.filter(F.col("ResponseId").rlike(numeric_regex)).select(
    "ResponseId",
    "Country",
    
    # YearsCode: If numeric, cast to int. If NULL or Junk, make it 0.
    F.when(F.col("YearsCode").rlike(numeric_regex), F.col("YearsCode").cast("int"))
     .otherwise(0).alias("YearsCode"),
    
    # YearsCodePro: Same logic, Junk/NULL becomes 0.
    F.when(F.col("WorkExp").rlike(numeric_regex), F.col("WorkExp").cast("int"))
     .otherwise(0).alias("YearsCodePro"),
    
    # Salary: If it looks like a float, cast it. Otherwise, leave as NULL.
    F.when(F.col("ConvertedCompYearly").rlike(float_regex), F.col("ConvertedCompYearly").cast("float"))
     .otherwise(None).alias("Salary")
)

df_tile1.select("YearsCode", "YearsCodePro", "Salary").show(5)

[Stage 37:=================================================>      (14 + 2) / 16]

+---------+------------+--------+
|YearsCode|YearsCodePro|  Salary|
+---------+------------+--------+
|       14|           8| 61256.0|
|       10|           2|104413.0|
|       12|          10| 53061.0|
|        5|           4| 36197.0|
|       22|          21| 60000.0|
+---------+------------+--------+
only showing top 5 rows


In [37]:
df_tile1.filter(F.col("YearsCode").isNull()).select("ResponseId", "YearsCode").show(5)

[Stage 40:====================================================>   (15 + 1) / 16]

+----------+---------+
|ResponseId|YearsCode|
+----------+---------+
+----------+---------+



In [38]:
df_tile1.select("YearsCode").summary().show()

[Stage 43:=================================================>      (14 + 2) / 16]

+-------+------------------+
|summary|         YearsCode|
+-------+------------------+
|  count|               938|
|   mean|19.662046908315563|
| stddev|11.966864107781308|
|    min|                 0|
|    25%|                10|
|    50%|                17|
|    75%|                27|
|    max|                53|
+-------+------------------+



In [39]:
df_tile1.show(10)

[Stage 46:=============================================>          (13 + 3) / 16]

+----------+-----------+---------+------------+--------+
|ResponseId|    Country|YearsCode|YearsCodePro|  Salary|
+----------+-----------+---------+------------+--------+
|         1|    Ukraine|       14|           8| 61256.0|
|         2|Netherlands|       10|           2|104413.0|
|         3|    Ukraine|       12|          10| 53061.0|
|         4|    Ukraine|        5|           4| 36197.0|
|         5|    Ukraine|       22|          21| 60000.0|
|         6|    Ukraine|       20|          15|120000.0|
|         7|    Ukraine|       13|           9|  6240.0|
|         8|    Ukraine|       30|          22| 72000.0|
|         9|    Ukraine|       15|           9| 70000.0|
|        10|    Ukraine|       10|           7|  5400.0|
+----------+-----------+---------+------------+--------+
only showing top 10 rows


In [41]:
df_tile1.filter(F.col("Salary").isNull()).select("ResponseId", "YearsCode", "YearsCodePro", "Country").show(5)

[Stage 52:====================================================>   (15 + 1) / 16]

+----------+---------+------------+-------+
|ResponseId|YearsCode|YearsCodePro|Country|
+----------+---------+------------+-------+
|        11|        9|           7|Ukraine|
|        14|        6|           6|  India|
|        15|        0|          15|Georgia|
|        16|        3|           5|  India|
|        18|        8|           1| Greece|
+----------+---------+------------+-------+
only showing top 5 rows


In [43]:
df_tech_raw = small_df.filter(F.col("ResponseId").rlike("^[0-9]+$")).select(
    "ResponseId",
    "DevType",
    "LanguageHaveWorkedWith",
    "DatabaseHaveWorkedWith"
)

df_tech_exploded = df_tech_raw.withColumn(
    "Language", F.explode(F.split(F.col("LanguageHaveWorkedWith"), ";"))
).filter(
    F.col("Language").isNotNull() & (F.col("Language") != "")
)

role_totals = df_tech_raw.filter(F.col("DevType").isNotNull()) \
    .groupBy("DevType") \
    .agg(F.countDistinct("ResponseId").alias("TotalDevsInRole"))

tech_counts = df_tech_exploded.filter(F.col("DevType").isNotNull()) \
    .groupBy("DevType", "Language") \
    .agg(F.countDistinct("ResponseId").alias("TechUsers"))

df_market_share = tech_counts.join(role_totals, "DevType") \
    .withColumn("MarketSharePct", (F.col("TechUsers") / F.col("TotalDevsInRole")) * 100) \
    .select(
        "DevType", 
        "Language", 
        F.round("MarketSharePct", 2).alias("MarketSharePct"),
        "TechUsers"
    ).orderBy("DevType", F.desc("MarketSharePct"))

In [44]:
role_totals.show(10)

[Stage 58:====================================================>   (15 + 1) / 16]

+--------------------+---------------+
|             DevType|TotalDevsInRole|
+--------------------+---------------+
|   Developer, mobile|             33|
| Developer, back-end|            149|
|Developer, front-end|             37|
| Engineering manager|             21|
|Developer, full-s...|            267|
|Architect, softwa...|             61|
|       Data engineer|             15|
|Financial analyst...|              3|
| Academic researcher|             24|
|Developer, embedd...|             26|
+--------------------+---------------+
only showing top 10 rows


In [45]:
tech_counts.show(10)

[Stage 61:====================================================>   (15 + 1) / 16]

+--------------------+--------------------+---------+
|             DevType|            Language|TechUsers|
+--------------------+--------------------+---------+
|   Developer, mobile|Bash/Shell (all s...|       11|
|   Developer, mobile|                Dart|       12|
|   Developer, mobile|                 SQL|       12|
| Developer, back-end|                Java|       53|
|Developer, front-end|                Dart|        2|
|Developer, front-end|            HTML/CSS|       33|
|Developer, front-end|          JavaScript|       34|
|Developer, front-end|          TypeScript|       26|
| Developer, back-end|              Kotlin|       16|
| Developer, back-end|                 SQL|       92|
+--------------------+--------------------+---------+
only showing top 10 rows


In [46]:
df_market_share.show(10)

+--------------+--------------------+--------------+---------+
|       DevType|            Language|MarketSharePct|TechUsers|
+--------------+--------------------+--------------+---------+
|AI/ML engineer|              Python|         86.67|       13|
|AI/ML engineer|          JavaScript|          60.0|        9|
|AI/ML engineer|Bash/Shell (all s...|          60.0|        9|
|AI/ML engineer|                 SQL|          60.0|        9|
|AI/ML engineer|            HTML/CSS|         53.33|        8|
|AI/ML engineer|                Java|          40.0|        6|
|AI/ML engineer|                  C#|          40.0|        6|
|AI/ML engineer|          TypeScript|          40.0|        6|
|AI/ML engineer|          PowerShell|         26.67|        4|
|AI/ML engineer|                   R|         26.67|        4|
+--------------+--------------------+--------------+---------+
only showing top 10 rows
